In [1]:
import pandas as pd
import numpy as np
import re
import html
import spacy

from pathlib import Path

In [2]:
DATA_PATH = Path("../data/processed/articles_metadata.csv")

articles_df = pd.read_csv(DATA_PATH)

articles_df["publication_date"] = pd.to_datetime(
    articles_df["publication_date"],
    errors="coerce"
)

print("Articles loaded:", len(articles_df))

articles_df.head()

Articles loaded: 18


,article_id,headline,publication_date,source,content
0,article_01,Heavy Rainfall Causes Flooding Across Riverton,2026-08-14,The Riverton Herald,Continuous rainfall began across Riverton late...
1,article_02,Schools Closed as Riverton River Level Rises,2026-08-14,National Daily News,Heavy rainfall continued in Riverton on Friday...
2,article_03,Power Supply Restored in Parts of Flood-Hit Ri...,2026-08-15,City News Network,"Rainfall eased in Riverton on Saturday, August..."
3,article_04,Temporary Shelters Open as Floodwater Enters R...,2026-08-14,Riverton Evening Post,Floodwater entered ground-floor homes in the R...
4,article_05,Old Bridge Closed to Vehicles as Riverton Rive...,2026-08-14,Metro Traffic Bulletin,The Old Bridge was closed to vehicles at aroun...


In [3]:
nlp = spacy.load("en_core_web_sm")

In [4]:
def clean_text(text):
    if pd.isna(text):
        return ""
    text = html.unescape(str(text))
    # Remove HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)
    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

In [5]:
articles_df["cleaned_content"] = (
    articles_df["content"].apply(clean_text)
)

articles_df[["article_id", "cleaned_content"]]

,article_id,cleaned_content
0,article_01,Continuous rainfall began across Riverton late...
1,article_02,Heavy rainfall continued in Riverton on Friday...
2,article_03,"Rainfall eased in Riverton on Saturday, August..."
3,article_04,Floodwater entered ground-floor homes in the R...
4,article_05,The Old Bridge was closed to vehicles at aroun...
5,article_06,The city weather office recorded 118 millimetr...
6,article_07,Authorities closed all municipal schools in Ri...
7,article_08,The district collector held a review meeting o...
8,article_09,Water began to recede from the Riverside neigh...
9,article_10,The Old Bridge remained closed while engineers...


In [6]:
sentence_records = []

for _, row in articles_df.iterrows():

    doc = nlp(row["cleaned_content"])

    for i, sent in enumerate(doc.sents):

        sentence_records.append({
            "article_id": row["article_id"],
            "publication_date": row["publication_date"],
            "sentence_id": i + 1,
            "sentence": sent.text.strip()
        })
sent_df = pd.DataFrame(sentence_records)
print("Total sentences:", len(sent_df))
sent_df.head(20)

Total sentences: 144


,article_id,publication_date,sentence_id,sentence
0,article_01,2026-08-14,1,Continuous rainfall began across Riverton late...
1,article_01,2026-08-14,2,The city weather office recorded 118 millimetr...
2,article_01,2026-08-14,3,Emergency services received their first major ...
3,article_01,2026-08-14,4,Water entered ground-floor homes in the Rivers...
4,article_01,2026-08-14,5,The municipal corporation opened two temporary...
5,article_01,2026-08-14,6,Officials said that 46 residents had been move...
6,article_01,2026-08-14,7,The weather office warned that further rainfal...
7,article_01,2026-08-14,8,Residents were advised to avoid underpasses an...
8,article_02,2026-08-14,1,Heavy rainfall continued in Riverton on Friday...
9,article_02,2026-08-14,2,The decision was announced at 7:15 a.m. after ...


In [7]:
OUTPUT_DIR = Path("../data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sent_df.to_csv(
    OUTPUT_DIR / "segmented_sentences.csv",
    index=False
)

print("Sentence dataset saved successfully!")

Sentence dataset saved successfully!


In [8]:
pos_records = []
for _, row in sent_df.iterrows():
    doc = nlp(row["sentence"])
    for token in doc:
        pos_records.append({
            "article_id": row["article_id"],
            "sentence_id": row["sentence_id"],
            "word": token.text,
            "lemma": token.lemma_,
            "pos": token.pos_,
            "tag": token.tag_,
            "dependency": token.dep_
        })
pos_df = pd.DataFrame(pos_records)
pos_df.head(30)

,article_id,sentence_id,word,lemma,pos,tag,dependency
0,article_01,1,Continuous,continuous,ADJ,JJ,amod
1,article_01,1,rainfall,rainfall,NOUN,NN,nsubj
2,article_01,1,began,begin,VERB,VBD,ROOT
3,article_01,1,across,across,ADP,IN,prep
4,article_01,1,Riverton,Riverton,PROPN,NNP,pobj
5,article_01,1,late,late,ADV,RB,advmod
6,article_01,1,on,on,ADP,IN,prep
7,article_01,1,August,August,PROPN,NNP,pobj
8,article_01,1,13,13,NUM,CD,nummod
9,article_01,1,",",",",PUNCT,",",punct


In [9]:
pos_distribution = pos_df["pos"].value_counts()

pos_distribution

pos
NOUN     565
VERB     299
ADP      244
DET      195
PUNCT    192
PROPN    189
AUX      123
ADJ      114
NUM       56
SCONJ     45
CCONJ     43
PART      42
ADV       40
PRON      22
Name: count, dtype: int64

In [10]:
pos_df.to_csv(
    OUTPUT_DIR / "pos_tagged_tokens.csv",
    index=False
)

print("POS annotations saved!")

POS annotations saved!


In [11]:
entity_records = []

for _, row in sent_df.iterrows():

    doc = nlp(row["sentence"])

    for ent in doc.ents:

        entity_records.append({
            "article_id": row["article_id"],
            "sentence_id": row["sentence_id"],
            "sentence": row["sentence"],
            "entity": ent.text,
            "entity_label": ent.label_
        })

entities_df = pd.DataFrame(entity_records)

entities_df.head(30)

,article_id,sentence_id,sentence,entity,entity_label
0,article_01,1,Continuous rainfall began across Riverton late...,Riverton,GPE
1,article_01,1,Continuous rainfall began across Riverton late...,August 13,DATE
2,article_01,1,Continuous rainfall began across Riverton late...,early Friday,DATE
3,article_01,1,Continuous rainfall began across Riverton late...,morning,TIME
4,article_01,2,The city weather office recorded 118 millimetr...,118,CARDINAL
5,article_01,2,The city weather office recorded 118 millimetr...,twelve hours,TIME
6,article_01,3,Emergency services received their first major ...,first,ORDINAL
7,article_01,3,Emergency services received their first major ...,approximately 5:30 a.m.,TIME
8,article_01,3,Emergency services received their first major ...,August 14,DATE
9,article_01,4,Water entered ground-floor homes in the Rivers...,Riverside,PERSON


In [12]:
entities_df.to_csv(
    OUTPUT_DIR / "named_entities.csv",
    index=False
)

print("Named entities saved!")

Named entities saved!
